# Prédiction de l'Éco-Score — Random Forest avec NLP

L'Éco-Score dépend majoritairement des **catégories, des ingrédients et des labels**. Nous allons donc utiliser un `TfidfVectorizer` pour transformer ces informations textuelles en données chiffrées utilisables par le modèle.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import learning_curve

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Chargement des datasets

In [ ]:
TARGET = 'environmental_score_grade'
FEATURES_TEXT = ['categories', 'ingredients_text'] # On limite aux deux plus importantes pour l'instant pour la mémoire
FEATURES_NUM = ['energy_100g', 'sugars_100g', 'saturated-fat_100g', 'salt_100g', 'fiber_100g', 'proteins_100g', 'fruits-vegetables-legumes_100g']

df_train = pd.read_csv('traite_avec_ecoscore.csv', low_memory=False)
df_pred = pd.read_csv('traite_sans_ecoscore.csv', low_memory=False)

for col in FEATURES_TEXT:
    df_train[col] = df_train[col].fillna('')
    df_pred[col] = df_pred[col].fillna('')

print(f"Entraînement : {len(df_train):,} produits")
print(f"A prédire : {len(df_pred):,} produits")
print("\nDistribution des grades :")
print(df_train[TARGET].value_counts())

In [ ]:
X = df_train[FEATURES_TEXT + FEATURES_NUM]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

## 2. Pipeline Scikit-Learn (NLP + RF)

Nous combinons les features de cette manière :
- `categories` : TF-IDF (100 mots max)
- `ingredients_text` : TF-IDF (200 mots max)
- `features numériques` : Imputation de la médiane

In [ ]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_tfidf', TfidfVectorizer(max_features=100), 'categories'),
        ('ing_tfidf', TfidfVectorizer(max_features=200), 'ingredients_text'),
        ('num_imputer', SimpleImputer(strategy='median'), FEATURES_NUM)
    ],
    remainder='drop'
)

# Pipeline finale
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

## 3. Entraînement et Évaluation

In [ ]:
# Entraînement sur le set de Train
print("Entraînement en cours... Cela peut prendre quelques minutes.")
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print("\nRapport de classification (Set de Test) :")
print(classification_report(y_test, y_pred))

## 4. Prédiction sur le Dataset sans Éco-Score

In [ ]:
print("Génération des prédictions pour les produits inconnus...")
X_new = df_pred[FEATURES_TEXT + FEATURES_NUM]
y_new_pred = pipeline.predict(X_new)

df_pred['environmental_score_grade_predicted'] = y_new_pred

print("\nRépartition des nouvelles prédictions :")
print(df_pred['environmental_score_grade_predicted'].value_counts())

df_pred.to_csv('produits_predictions_ecoscore.csv', index=False)
print("\n✅ Prédictions sauvegardées dans 'produits_predictions_ecoscore.csv'")